# RecSys 2026 — LoRA Fine-tune for Blind-A (Colab notebook)

Runs the full pipeline on Colab GPU.

## Setup checklist

1. **Runtime → Change runtime type → A100 GPU** (best). T4/L4 also work, just slower.
2. Upload this entire repo (the `recsys2026-lora-tutorial/` directory) to your Drive at `/content/drive/MyDrive/recsys2026-lora-tutorial/` — OR clone from GitHub if you've pushed it.
3. Run all cells. The final cell zips the adapter and copies it to Drive.

Total wall time on A100: ~50 min. The trained adapter zip is ~80 MB.

In [ ]:
# Verify GPU.
!nvidia-smi | head -20

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Stage repo from Drive into Colab's local filesystem (fast disk).
import os, shutil
DRIVE_REPO = '/content/drive/MyDrive/recsys2026-lora-tutorial'
WORK = '/content/recsys2026-lora-tutorial'
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(DRIVE_REPO, WORK)
print('staged at', WORK)
!ls {WORK}

In [ ]:
# Install pinned deps (Colab usually has torch already; we let pip resolve).
%cd {WORK}
!pip install -q -r requirements.txt
!python -c "import torch, transformers, trl, peft, bm25s; print('torch', torch.__version__, 'transformers', transformers.__version__, 'trl', trl.__version__, 'peft', peft.__version__)"

In [ ]:
# Run the full pipeline. Skip venv (we use Colab's Python directly) and 
# skip the inference + zip stages — those will run on your local M4 with
# the downloaded adapter. Bumps to A100-friendly hyperparams.
%cd {WORK}
!SKIP_VENV=1 \
  SKIP_INFERENCE=1 \
  SKIP_ZIP=1 \
  PER_DEVICE_BATCH=2 \
  GRAD_ACCUM=8 \
  TRAIN_MAX_LENGTH=3072 \
  bash run_pipeline.sh

In [ ]:
# Zip the adapter + push to Drive.
import shutil, os
ADAPTER = f'{WORK}/lora_adapters/qwen3b_blinda_v1/final_adapter'
OUT_ROOT = '/content/drive/MyDrive/recsys2026-lora-tutorial/adapters'
os.makedirs(OUT_ROOT, exist_ok=True)
zip_base = f'{OUT_ROOT}/qwen3b_blinda_v1'
shutil.make_archive(zip_base, 'zip', ADAPTER)
print('wrote', zip_base + '.zip')
!ls -lh {zip_base}.zip

## Done. On your M4:

```bash
cd recsys2026-lora-tutorial
mkdir -p lora_adapters/qwen3b_blinda_v1/final_adapter
unzip ~/Downloads/qwen3b_blinda_v1.zip -d lora_adapters/qwen3b_blinda_v1/final_adapter

# Run inference + package only (training was done in Colab).
SKIP_DATASET=1 SKIP_TRAIN=1 ./run_pipeline.sh
```

`output/prediction.zip` will be ready to upload to CodaBench.